# Final Project - Kansas HMDA Home-Purchase Applications
Scott Person

In [0]:
data_dir = './data/'

In [0]:
# imports
import os
import glob
import pandas as pd

In [0]:
# Setting this to Y reloads all of the data from the source files, trims to necessary analysis columns, and saves the smaller dataset to a CSV. N loads the analysis CSV.
RELOAD_YEARLY_FILES = 'N'

## Project Overview

My Question: Among Kansas home-purchase applications that received a lending decision from 2018 to 2025, were denial rates higher in lower-income census tracts than in higher-income tracts, even when comparing applicants within similar income bands?


| Part | What to write down |
|---|---|
| **Unit** | One row represents a mortgage application event. |
| **Outcome** | We are looking only at applications with a lending decision so action_taken codes 1,2,3. Our numerator will be the count of rows with action_taken = 3 (denied). The denominator will be the sum of counts with acti    on_taken of 1,2,3 (originated, approved but not accepted by applicant, denied).|
| **Groups** | We will be using the tract_to_msa_income_percentage divided into two bands: at or above median, and below median income for the MSA. We will also be dividing the loans by applicant income bands.|
| **Time** | activity_year |
| **Either way?** | I would be surprised if lower-income tracts had equal or lower denial rates than higher-income tracts within the same applicant-income bands.|


Source for action_taken codes: https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields/#action_taken

## Initial Data Review

In [0]:
if RELOAD_YEARLY_FILES == 'Y':
    csv_files = glob.glob(os.path.join(data_dir, "hmda*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")


    frames = []
    for csv_path in csv_files:
        file_name = os.path.basename(csv_path)
        table_name = os.path.splitext(file_name)[0]
        print(f"Loading {file_name} ...")
        file_df = pd.read_csv(csv_path, dtype={"total_units": "string"}) #The total_units column contains text and numbers in the 2019 file. I won't be using this column anyway.

        print(f"  -> {table_name}: {file_df.shape[0]} rows, {file_df.shape[1]} columns")
        frames.append(file_df)

    df_all = pd.concat(frames, ignore_index=True)

    analysis_columns = ["action_taken", "tract_to_msa_income_percentage", "income", "activity_year", "derived_msa-md", "state_code", "loan_purpose"]

    df = df_all[analysis_columns].copy()

    df.to_csv(os.path.join(data_dir, "analysis_data.csv"), index=False)
    print(f"Wrote analysis_data.csv with {df.shape[0]} rows, {df.shape[1]} columns")
else:
    df = pd.read_csv(os.path.join(data_dir, "analysis_data.csv"))


In [0]:
df.head()

In [0]:
def to_snake(columns):
    'Lowercase, turn anything that is not a letter or digit into _, trim stray underscores.'
    return (pd.Index(columns)
            .str.strip()
            .str.lower()
            .str.replace(r"[^0-9a-z]+", "_", regex=True)
            .str.strip("_"))

In [0]:
df_columns_orig = df.columns
df.columns = to_snake(df_columns_orig)

pd.DataFrame({"before": df_columns_orig, "after": df.columns})

In [0]:
def first_look(df, name="dataset", max_categories=12):
    'Run the standard first-look routine on a DataFrame.'
    print("=" * 62)
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("=" * 62)
 
    print("\n--- TYPES AND MISSING ---")
    summary = pd.DataFrame({
        "dtype":    df.dtypes.astype(str),
        "missing":  df.isna().sum(),
        "missing%": (df.isna().mean() * 100).round(1),
        "unique":   df.nunique(),
    })
    print(summary)
 
    numeric = df.select_dtypes("number")
    if not numeric.empty:
        print("\n--- NUMERIC SUMMARY ---")
        print(numeric.describe().round(2))
 
    text = df.select_dtypes("object")
    if not text.empty:
        print("\n--- TEXT / CATEGORY COLUMNS ---")
        for col in text.columns:
            n = df[col].nunique()
            if n <= max_categories:
                top = df[col].value_counts().to_dict()
                print(f"{col} ({n} values): {top}")
            else:
                print(f"{col} ({n} values, too many to list) e.g. {df[col].dropna().iloc[0]!r}")
 
    dupes = df.duplicated().sum()
    print(f"\n--- DUPLICATE ROWS: {dupes} ---")
    return summary

The to_snake function made one very minor change: derived_msa-md to derived_msa_md. My assumption is that the folks upstream and FFIEC know about snake case and proper data formating as well. Thank you Federal Government employees!

As mentioned above, I cut down the columns from 99 provided in the full set to the seven that I think I'll need for the analysis. One row represents one mortgage loan application.

In [0]:
first_look(df)

### Basic Calculation
Below I took the raw data and performed the calculations that I expect will be roughly the same shape as my eventual data output.

In [0]:
df["tract_group"] = df["tract_to_msa_income_percentage"].apply(
    lambda x: "At or above median" if x >= 100 else "Below median"
)

df["income_band"] = pd.cut(
    df["income"],
    bins=[-float("inf"), 50, 100, 150, float("inf")],
    right=False,
    labels=["Under $50k", "$50k–<$100k", "$100k–<$150k", "$150k+"]
).astype("string").fillna("Missing")

raw_result = df.groupby(["income_band", "tract_group"])["action_taken"].agg(
    applications="size",
    denials=lambda codes: (codes == 3).sum()
)
raw_result["denial_rate_pct"] = 100 * raw_result["denials"] / raw_result["applications"]
print(raw_result.round(2))

## Data Integrity Audit

### Look for missing data

In [0]:
rows = []

for column in df.columns:
    text = df[column].astype("string").str.strip()
    lower = text.str.lower()

    flags = {
        "null": text.isna(),
        "blank": text.eq(""),
        "?": text.eq("?"),
        "NA": lower.eq("na"),
        "Exempt": lower.eq("exempt"),
        "zero": pd.to_numeric(text, errors="coerce").eq(0),
    }
    flags = {name: mask.fillna(False) for name, mask in flags.items()}
    any_flag = pd.concat(flags.values(), axis=1).any(axis=1)

    rows.append({
        "column": column,
        **{name: int(mask.sum()) for name, mask in flags.items()},
        "flagged_pct": round(100 * any_flag.mean(), 2),
    })

missing_audit = pd.DataFrame(rows)
display(missing_audit)

### Duplicates
We see a high number of duplicates (219126). This isn't really an issue here. I'm confident in the data load and we only have seven broad columns. There isn't a good natural key in the base data (like loan or application id) that we could use to identify true duplicates.

In [0]:
duplicate_rows = df[df.duplicated(keep=False)]
print(f"Duplicate row count: {len(duplicate_rows)}")
duplicate_rows.head(20)

### Data Types
This is a pretty easy fix. Change MSA to a string, state to a string, and action_taken and loan_purpose to categories.

In [0]:
df.info()

### Other Problems
* I need to drop all rows with action taken not in ["1","2","3"]. Should probably do that BEFORE creating the category.
* There is an interesting MSA of all 9s. This one is apprently for tracts outside of a Metropoliton Statistical Area (MSA). Source: https://www.ffiec.gov/sites/default/files/data/census/cen2024_16JLUY24.pdf. My intention is to analyze these separately.

## Cleaning & Remediation

### Missing Data

In [0]:
msa_counts = (
    df[df["tract_to_msa_income_percentage"] == 0]
    ["derived_msa_md"]
    .dropna()
    .astype("string")
    .value_counts()
    .sort_index()
    .reset_index()
)
msa_counts.columns = ["derived_msa_md", "count"]
msa_counts

I don't have an alternative to impute those values so I'm going to drop them.

In [0]:
before_count = len(df)
df = df[df["tract_to_msa_income_percentage"] != 0].copy()
after_count = len(df)

print(f"Rows before: {before_count:,}")
print(f"Rows after:  {after_count:,}")
print(f"Dropped:     {before_count - after_count:,}")

This had the effect of dropping all 872 records that had derived_msa_md == 0.

In [0]:
df[df["derived_msa_md"] == 0]

In [0]:
print(f"Income min: {df['income'].min()}")
print(f"Income max: {df['income'].max()}")

The HMDA filing instructions indicate that negative and positive values are acceptible for income. We will therefore retain the negative values. Also the income is reported in thousands so 2,134,000 is $2.1B. That is probably not accurate but even 2,134 (excluding three zeros) would put it in the planned upper income band so we will just retain that value.

Source: https://ffiec.cfpb.gov/documentation/fig/2025/overview

### Drop non-1,2,3 action_taken records

In [0]:
before_count = len(df)
valid_actions = [1, 2, 3]
df = df[df["action_taken"].isin(valid_actions)].copy()
after_count = len(df)

print(f"Rows before: {before_count:,}")
print(f"Rows after:  {after_count:,}")
print(f"Dropped:     {before_count - after_count:,}")

### Missing Income Values
Our analysis is based on income bands. Records with missing income therefore have no analytical value. This represents about 5% of the records. These will be dropped.

In [0]:
before_count = len(df)
df = df[df["income"].notna()].copy()
after_count = len(df)

print(f"Rows before: {before_count:,}")
print(f"Rows after:  {after_count:,}")
print(f"Dropped:     {before_count - after_count:,}")

### Duplicates
We see a high number of duplicates (219126). This isn't really an issue here. I'm confident in the data load and we only have seven broad columns. There isn't a good natural key in the base data (like loan or application id) that we could use to identify true duplicates.

### Data Types
This is a pretty easy fix. Change MSA to a string, state to a string, and action_taken and loan_purpose to categories.

In [0]:
df.info()

In [0]:
df["derived_msa_md"] = df["derived_msa_md"].astype("string")
df["state_code"] = df["state_code"].astype("string")
df["action_taken"] = df["action_taken"].astype("category")
df["loan_purpose"] = df["loan_purpose"].astype("category")

In [0]:
df.info()

### Cleaning Decision Log

| Column                           | What I saw | What I did | What it cost |
|----------------------------------|------------|------------|--------------|
| `action_taken`|Records with indeterminate lending decision (not 1,2,3)|All records not 1,2,3 were removed from the dataframe|None|
| `tract_to_msa_income_percentage` |Values == 0|Dropped the relevant records| 8442 records dropped; they were of no analytical value so no cost.|
| `income`|We have records with zero and negative income|Nothing|These values are allowable according to the instructions. We are not planning to aggregate income but if we do the most likely function is median, which will mitigate the impact of the outliers.|
| `income`|17,910 records with no income value|These records were dropped|Lost 5.4% of decided applications, including 875 denials; the overall denial rate rose from 9.10% to 9.34%. Findings within income bands may not represent applications with missing income.|
| `derived_msa_md`|872 records had a value of zero|Nothing|None. The tract_to_msa_income_percentage mitigation removed all of thse records.              |

In [0]:
df["tract_group"] = df["tract_to_msa_income_percentage"].apply(
    lambda x: "At or above median" if x >= 100 else "Below median"
)

df["income_band"] = pd.cut(
    df["income"],
    bins=[-float("inf"), 50, 100, 150, float("inf")],
    right=False,
    labels=["Under $50k", "$50k–<$100k", "$100k–<$150k", "$150k+"]
).astype("string").fillna("Missing")

clean_result = df.groupby(["income_band", "tract_group"])["action_taken"].agg(
    applications="size",
    denials=lambda codes: (codes == 3).sum()
)
clean_result["denial_rate_pct"] = 100 * clean_result["denials"] / clean_result["applications"]

print('**********Raw Result**********')
print(raw_result.round(2))
print('')
print('**********Clean Result**********')
print(clean_result.round(2))

We observe somewhat higher denial rates with the clean data. This is possibly due to the decrease in the denominator (size) of the analytical dataset following the removal of records that did not have a loan determination. A lower denominator will result in a larger quotient.